# 5주차 핵심 기능 복습

1–5차시의 핵심 기능 복습과 프로젝트 명세 작성용 Notebook입니다. 강의의 서비스 사례는 응용 설계 예시이며 모두 구현된 제품 목록은 아닙니다.

저장소 안에서 VS Code로 열고 Python 커널을 선택하세요. 실습 데이터는 합성 예시입니다.


In [ ]:
%pip install "pydantic>=2,<3" "pytest>=8,<9" "openpyxl>=3.1.5,<4"


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/day1_agent.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from pathlib import Path
from src.course_services.service_router import route_service_request
from src.course_services.meeting_service import validate_action_evidence
from src.course_services.review_service import run_review_service
from labs.day4.office_lab.sheets.student_excel import (
    load_sample, calculate_wbs, validate_wbs)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/day1_agent.py").exists())
from IPython.display import Markdown, display


## 1차시. 업무 자동화의 공통 구조

사용자의 요청이 같아도 필요한 자동화의 범위는 다릅니다.

실행 순서: 복습 Notebook의 1차시 셀 실행 / meeting_transcript를 unknown으로 변경 / 중단 이유가 표시되는지 확인 / 원래 입력 종류로 복구 후 다시 실행


In [ ]:
result = route_service_request(
    input_kind="meeting_transcript",
    source_path=ROOT / "data/meeting_sample_ko.txt",
    workspace_root=ROOT,
)
print(result["service"], result["status"])
assert result["status"] == "SUCCESS"
assert result["service"] == "meeting"
assert result["external_write"] is False


In [ ]:
blocked = route_service_request(
    input_kind="unknown",
    source_path=ROOT / "data/meeting_sample_ko.txt",
    workspace_root=ROOT,
)
assert blocked["error_code"] == "UNSUPPORTED_INPUT_KIND"


In [ ]:
display(Markdown(f"**회의 처리 결과**\n\n서비스: {result["service"]} / 상태: {result["status"]}\n\n외부 반영: {result["external_write"]}"))


## 2차시. 음성 기록과 업무 인계

전사는 발화를 옮기는 작업이고, 요약은 그 안에서 필요한 내용을 고르는 작업입니다.

실행 순서: 2차시의 원문 s01과 s02 확인 / 근거 번호 s01로 검사 실행 / 존재하지 않는 s99로 변경 / 오류 확인 후 원문 번호로 복구


In [ ]:
segments = [
    {"id": "s01", "text": "민지가 문구를 수정합니다."},
    {"id": "s02", "text": "기한은 아직 미정입니다."},
]
actions = [{"task": "문구 수정", "evidence_ids": ["s01"]}]
errors = validate_action_evidence(
    actions, known_segment_ids={"s01", "s02"})
assert errors == []
assert actions[0]["evidence_ids"] == ["s01"]


In [ ]:
actions[0]["evidence_ids"] = ["s99"]
errors = validate_action_evidence(
    actions, known_segment_ids={"s01", "s02"})
assert errors == ["ACTION_1_UNKNOWN_EVIDENCE:s99"]


In [ ]:
display(Markdown("**근거 검사 결과**\n\n" + "\n".join(f"- {e}" for e in errors)))


## 3차시. 변경 검토와 수정 제안

변경 검토는 이전 내용과 달라진 위치, 그로 인한 영향을 함께 설명합니다.

실행 순서: 3차시의 샘플 Diff 읽기 / 검토 의견의 9, 11, 12행 확인 / 빈 Diff로 다시 실행 / 빈 입력의 오류 코드 확인


In [ ]:
diff_text = (ROOT /
    "data/day3_review_cases/unsafe_pr.diff").read_text()
report = run_review_service(diff_text)
for item in report["findings"]:
    print(item["line"], item["title"])
assert report["status"] == "SUCCESS"
assert [f["line"] for f in report["findings"]] == [9, 11, 12]
assert report["automatic_publish"] is False


In [ ]:
empty_review = run_review_service("")
assert empty_review["status"] == "EXPECTED_FAILURE"
assert empty_review["error_code"] == "EMPTY_DIFF"


In [ ]:
display(Markdown("| 행 | 검토 의견 |\n|---:|---|\n" + "\n".join(f"| {f["line"]} | {f["title"]} |" for f in report["findings"])))


## 4차시. 문서와 표의 재사용

원본 값과 계산 규칙을 먼저 정하면 여러 문서의 내용을 함께 갱신할 수 있습니다.

실행 순서: 4차시 일정표의 날짜와 진행률 확인 / 진행률 한 건을 1.0으로 변경 / 완료 상태로 바뀌는지 실행 확인 / 1.5 입력으로 오류 재현


In [ ]:
tasks = load_sample("wbs_tasks.json")
rows = calculate_wbs(tasks, "2026-09-20")
for row in rows:
    print(row["id"], row["working_days"], row["status"])
assert validate_wbs(tasks) == []
assert len(rows) == len(tasks)
assert all(row["working_days"] >= 0 for row in rows)


In [ ]:
broken = [dict(task) for task in tasks]
broken[0]["progress"] = 1.5
assert any(e["code"] == "INVALID_PROGRESS"
           for e in validate_wbs(broken))


In [ ]:
display(Markdown("| 업무 | 평일 수 | 상태 |\n|---|---:|---|\n" + "\n".join(f"| {r["task"]} | {r["working_days"]} | {r["status"]} |" for r in rows)))


## 5차시. 업무 서비스와 프로젝트 선택

한 사람이 실제 입력을 넣고 원하는 결과를 확인하는 한 번의 사용 과정을 정합니다.

실행 순서: 5차시의 사용자와 입력을 내 업무로 변경 / 예상 결과를 화면 또는 파일로 명시 / 실패할 수 있는 상황 두 가지 작성 / Codex 요청문에 완료 조건으로 사용


In [ ]:
project = {
    "user": "고객 의견을 정리하는 기획자",
    "input": "익명 후기 CSV",
    "result": "주제별 건수와 대표 원문",
    "failure_cases": ["빈 파일", "필수 열 없음"],
    "external_write": False,
}
assert project["user"] and project["input"]
assert project["result"]
assert len(project["failure_cases"]) == 2


In [ ]:
assert project["external_write"] is False
# 이후 구현할 CSV 검사는 별도 테스트로 확인
# 이 셀은 프로젝트 명세의 필수 항목만 검사


In [ ]:
display(Markdown("**내 프로젝트 명세**\n\n" + "\n".join(f"- {k}: {v}" for k,v in project.items())))


## 6–7차시. 개인 프로젝트 제작

앞에서 작성한 명세를 자신의 프로젝트에 적용합니다. 강사의 별도 서비스 시연 기획서, 소스코드, 데이터는 이 배포본에 포함하지 않습니다.

1. 첫 입력과 실제 결과 확인
2. 예상한 실패 상황 재현
3. README에 실행 명령 기록
4. 다음 개선 한 가지 선택
